In [ ]:
from pathlib import Path
import sys

src_dir = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "common.py").is_file() and (p / "feature_preference").is_dir())
sys.path.insert(0, str(src_dir))


In [ ]:
from torchvision.transforms.v2.functional import to_pil_image

%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import torch.nn.functional as F
from common import *

In [ ]:
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})
colors_darker = sns.color_palette("dark", 5 * 2)
colors = sns.color_palette("deep", 5 * 2)
colors

In [ ]:
import os


def load_from_csv(path):
    output = []
    if not os.path.exists(path):
        return np.array(output)
    with open(path, 'r') as f:
        for line in f:
            output.append(float(line.strip()))
    return np.array(output)


def load_from_csv_split(path):
    output = []
    if not os.path.exists(path):
        return np.array(output)
    with open(path, 'r') as f:
        for line in f:
            split = line.strip().split(',')
            if len(output) == 0:
                for i in range(len(split)):
                    output.append([])
            for i in range(len(split)):
                output[i].append(float(split[i].strip()))
    for i in range(len(output)):
        output[i] = np.array(output[i])
    return output

In [ ]:
# ratios = [0.05, 0.075, 0.1, 0.2]
ratios = [0.03, 0.06, 0.09]
lambdas = 0.01
loss = []
acc_object = []
acc_background = []
steps = []
eig_list = []
method = "BarlowTwins_multiple_filtered"
for seed in range(6):
    loss.append([])
    acc_object.append([])
    acc_background.append([])
    steps.append([])
    eig_list.append([])
    directory_models = f"/models/jhkim/waterbirds/{method}_{lambdas}/{seed}"
    directory = f"datas/{method}_{lambdas}/{seed}"
    # directory = f"datas/lambda_{method}_{lambdas}"
    for r in ratios:
        loss_array = load_from_csv(f"{directory}/{r}/loss.csv")
        [acc_object_array, acc_background_array, step_array] = load_from_csv_split(f"{directory}/{r}/accuracy.csv")
        eigvals_lists_array = load_from_csv_split(f"{directory}/{r}/eig.csv")
        loss[-1].append(loss_array)
        acc_object[-1].append(acc_object_array)
        acc_background[-1].append(acc_background_array)
        steps[-1].append(step_array)
        eig_list[-1].append(eigvals_lists_array)

In [ ]:
# for r_idx, r_val in enumerate(ratios):
#     lengths = [len(steps[s_idx][r_idx]) for s_idx in range(6)]
#     max_len = max(lengths)
#     min_len = min(lengths)
#
#     if min_len != max_len:
#         print(f"Ratio {r_val}: Data insufficient. Min {min_len} < Target {max_len}")

fig, axs = plt.subplots(1, 2, figsize=(28, 10))

for r_idx, r_val in enumerate(ratios):

    # obj_acc_np = np.array([a[:plot_len] for a in obj_acc_list])
    # bg_acc_np = np.array([a[:plot_len] for a in bg_acc_list])
    # current_steps = step_list[:plot_len]

    current_steps = steps[0][r_idx]
    obj_acc_np = acc_object[0][r_idx]
    # obj_std = np.std(obj_acc_np, axis=0)
    # bg_mean = np.mean(bg_acc_np, axis=0)

    axs[0].plot(current_steps, obj_acc_np, color=colors_darker[r_idx], linewidth=5, alpha=0.9, label=f"Object ratio : {r_val:.3f}")
    # axs[0].fill_between(current_steps, obj_mean - obj_std, obj_mean + obj_std, color=colors_darker[r_idx], alpha=0.2)
    # axs[0].plot(current_steps, bg_mean, linestyle=":", color=colors[r_idx], linewidth=4, alpha=0.75, label=f"Bg ratio : {r_val:.3f}")
# for r_idx, r_val in enumerate(ratios):
#     plot_len = min([len(steps[s_idx][r_idx]) for s_idx in range(6)])
#
#     obj_acc_list = [acc_object[s_idx][r_idx] for s_idx in range(6)]
#     bg_acc_list = [acc_background[s_idx][r_idx] for s_idx in range(6)]
#     step_list = steps[0][r_idx]
#
#     obj_acc_np = np.array([a[:plot_len] for a in obj_acc_list])
#     bg_acc_np = np.array([a[:plot_len] for a in bg_acc_list])
#     current_steps = step_list[:plot_len]
#
#     obj_mean = np.mean(obj_acc_np, axis=0)
#     obj_std = np.std(obj_acc_np, axis=0)
#     bg_mean = np.mean(bg_acc_np, axis=0)
#
#     axs[0].plot(current_steps, obj_mean, color=colors_darker[r_idx], linewidth=5, alpha=0.9, label=f"Object ratio : {r_val:.3f}")
#     axs[0].fill_between(current_steps, obj_mean - obj_std, obj_mean + obj_std, color=colors_darker[r_idx], alpha=0.2)
#     axs[0].plot(current_steps, bg_mean, linestyle=":", color=colors[r_idx], linewidth=4, alpha=0.75, label=f"Bg ratio : {r_val:.3f}")

axs[0].set_xlabel("Step")
axs[0].set_ylabel("Test Accuracy")
axs[0].set_ylim(30, 100)
axs[0].legend(loc='lower right')

scatter_data = {ratio: {'steps': [], 'seeds': []} for ratio in ratios_scatter}

for seed in tqdm(seeds):
    directory = f"datas/{method}_{lambdas}/{seed}"

    for ratio in ratios_scatter:
        try:
            [acc_object_array, acc_background_array, step_array] = load_from_csv_split(f"{directory}/{ratio}/accuracy.csv")

            if len(step_array) > 0:
                step_at_target = step_array[-1]
                scatter_data[ratio]['steps'].append(step_at_target)
                scatter_data[ratio]['seeds'].append(seed)
        except Exception as e:
            print(f"Error loading seed {seed}, ratio {ratio}: {e}")
            continue

means_x = []
means_y = []
stds_y = []

for i, ratio in enumerate(ratios_scatter):
    if len(scatter_data[ratio]['steps']) > 0:
        seeds_list = scatter_data[ratio]['seeds']
        steps_list = scatter_data[ratio]['steps']

        max_seed = max(seeds_list)

        completed_x = []
        completed_y = []
        running_x = []
        running_y = []

        for seed, step in zip(seeds_list, steps_list):
            completed_x.append(ratio)
            completed_y.append(step)
        if len(completed_x) > 0:
            axs[1].scatter(completed_x, completed_y, s=150, alpha=0.25, color=colors[i],)
                       # label=f'Object Ratio : {ratio:.2f}'

        if len(completed_y) > 0:
            steps_arr = np.array(completed_y)
            mean_step = steps_arr.mean()
            std_step = steps_arr.std()
            means_x.append(ratio)
            means_y.append(mean_step)
            stds_y.append(std_step)

if len(means_x) > 0:
    means_x_arr = np.array(means_x)
    means_y_arr = np.array(means_y)
    stds_y_arr = np.array(stds_y)
    axs[1].plot(means_x_arr, means_y_arr, color='red', linewidth=3, alpha=0.8, marker='o')
    axs[1].fill_between(means_x_arr,
                    means_y_arr - stds_y_arr,
                    means_y_arr + stds_y_arr,
                    alpha=0.12, color='red')

axs[1].set_xlabel('Object Ratio')
axs[1].set_ylabel('Training step at 80% accuracy')
# ax.set_title('Last Recorded Steps vs Object Ratio (Multiple Seeds)')
# ax.set_yscale('log')
# axs[1].legend()
axs[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'figure/scatter_ratio_vs_steps.pdf', format='pdf')
plt.show()

In [ ]:
for r in range(len(ratios)):
    for i in range(len(eig_list[r])):
        plt.plot(steps[r][:len(eig_list[r][i])], eig_list[r][i], linewidth=2.0)
    plt.yscale('log')
    plt.xlabel("Step")
    plt.ylabel("C Eigenvalue")
    plt.title(f"Object ratio : {ratios[r]:.3f}")
    # plt.ylim(1e-3, 1e2)
    plt.show()

In [ ]:
for r in range(len(ratios)):
    plt.plot(loss[r], label=f"ratio: {ratios[r]}", color=colors[r], alpha=0.6, linewidth=3)
    plt.xlabel("Step")
    plt.ylabel("Loss")
# plt.ylim(100, 300)
# plt.yscale('log')
plt.legend()
plt.show()

In [ ]:
seeds = range(0, 6)
# ratios_scatter = [0.03, 0.04, 0.05, 0.07, 0.09]
ratios_scatter = [0.03 + 0.01*i for i in range(0, 7)]
lambdas = 0.01
method = "BarlowTwins_multiple_filtered"
target_accuracy = 80.0

scatter_data = {ratio: {'steps': [], 'seeds': []} for ratio in ratios_scatter}

for seed in tqdm(seeds):
    directory = f"datas/{method}_{lambdas}/{seed}"
    
    for ratio in ratios_scatter:
        try:
            [acc_object_array, acc_background_array, step_array] = load_from_csv_split(f"{directory}/{ratio}/accuracy.csv")
            
            if len(step_array) > 0:
                step_at_target = step_array[-1]
                scatter_data[ratio]['steps'].append(step_at_target)
                scatter_data[ratio]['seeds'].append(seed)
        except Exception as e:
            print(f"Error loading seed {seed}, ratio {ratio}: {e}")
            continue
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

means_x = []
means_y = []
stds_y = []

for i, ratio in enumerate(ratios_scatter):
    if len(scatter_data[ratio]['steps']) > 0:
        seeds_list = scatter_data[ratio]['seeds']
        steps_list = scatter_data[ratio]['steps']

        max_seed = max(seeds_list)

        completed_x = []
        completed_y = []
        running_x = []
        running_y = []

        for seed, step in zip(seeds_list, steps_list):
            completed_x.append(ratio)
            completed_y.append(step)
            # if seed == max_seed:
            #     running_x.append(ratio)
            #     running_y.append(step)
            # else:
            #     completed_x.append(ratio)
            #     completed_y.append(step)

        if len(completed_x) > 0:
            ax.scatter(completed_x, completed_y, s=150, alpha=0.25, color=colors[i],
                       label=f'Ratio {ratio:.2f}')

        # if len(running_x) > 0:
        #     ax.scatter(running_x, running_y, s=150, alpha=0.15, color=colors[i],
        #                marker='x', linewidths=3, label=f'Ratio {ratio:.2f} (running)')

        if len(completed_y) > 0:
            steps_arr = np.array(completed_y)
            mean_step = steps_arr.mean()
            std_step = steps_arr.std()
            means_x.append(ratio)
            means_y.append(mean_step)
            stds_y.append(std_step)

if len(means_x) > 0:
    means_x_arr = np.array(means_x)
    means_y_arr = np.array(means_y)
    stds_y_arr = np.array(stds_y)
    ax.plot(means_x_arr, means_y_arr, color='red', linewidth=3, alpha=0.8, marker='o')
    ax.fill_between(means_x_arr,
                    means_y_arr - stds_y_arr,
                    means_y_arr + stds_y_arr,
                    alpha=0.12, color='red')

ax.set_xlabel('Object Ratio')
ax.set_ylabel('Training step at 80% accuracy')
# ax.set_title('Last Recorded Steps vs Object Ratio (Multiple Seeds)')
# ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'figure/scatter_ratio_vs_steps.pdf', format='pdf')
plt.show()

print(f"\n=== Statistics for Last Recorded Steps (excluding running seed) ===")
for ratio in ratios_scatter:
    if len(scatter_data[ratio]['steps']) > 0:
        seeds_list = scatter_data[ratio]['seeds']
        steps_list = scatter_data[ratio]['steps']
        
        max_seed = max(seeds_list)
        completed_steps = [step for seed, step in zip(seeds_list, steps_list) if seed != max_seed]
        
        if len(completed_steps) > 0:
            steps = np.array(completed_steps)
            print(f"\nRatio {ratio:.3f}:")
            print(f"  Mean steps: {steps.mean():.2f}")
            print(f"  Std steps: {steps.std():.2f}")
            print(f"  Min steps: {steps.min():.2f}")
            print(f"  Max steps: {steps.max():.2f}")
            print(f"  Number of completed seeds: {len(steps)}")
            print(f"  Running seed ({max_seed}): {[step for seed, step in zip(seeds_list, steps_list) if seed == max_seed]}")
        else:
            print(f"\nRatio {ratio:.3f}:")
            print(f"  No completed seeds (all running)")
            print(f"  Running seed ({max_seed}): {steps_list}")


In [ ]:
import torchvision.transforms.v2 as transforms
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
from PIL.Image import Image
from PIL import Image, ImageOps, ImageFilter
import pandas as pd
import torchvision
import cv2

device = torch.device("cuda")
size = 128


class WaterbirdDataset(Dataset):
    def __init__(self, root_dir, metadata_path, split='train', transform=None, balance_classes=True, no_aug=False):
        self.root_dir = root_dir
        self.transform = transform
        self.split = split
        self.no_aug = no_aug
        self.metadata = pd.read_csv(metadata_path)

        split_dict = {'train': 0, 'val': 1, 'test': 2}

        self.metadata = self.metadata[self.metadata['split'] == split_dict[split]]
        if balance_classes:
            landbird_data = self.metadata[self.metadata['y'] == 0]
            waterbird_data = self.metadata[self.metadata['y'] == 1]

            min_class_size = min(len(landbird_data), len(waterbird_data))
            # min_class_size = 256

            landbird_balanced = landbird_data.sample(min_class_size, random_state=1)
            waterbird_balanced = waterbird_data.sample(min_class_size, random_state=1)

            self.metadata = pd.concat([landbird_balanced, waterbird_balanced])

        self.metadata = self.metadata.reset_index(drop=True)

        self.size_transform = transforms.Resize(size)

        self.labels = []
        self.places = []
        self.img_ids = []
        self.img = []
        mean = 0.0
        var = 0.0

        for idx in range(len(self.metadata)):
            img_filename = self.metadata.loc[idx, 'img_filename']
            self.labels.append(int(self.metadata.loc[idx, 'y']))
            self.places.append(int(self.metadata.loc[idx, 'place']))
            self.img_ids.append(self.metadata.loc[idx, 'img_id'] if 'img_id' in self.metadata.columns else idx)
            image = Image.open(os.path.join(self.root_dir, img_filename)).convert('RGB')
            image_resize = self.size_transform(image)
            self.img.append(image_resize)
            image_preprocess = transforms.ToTensor()(image_resize)
            mean += image_preprocess.mean(dim=[1, 2])
            var += image_preprocess.var(dim=[1, 2])

        mean /= len(self.img)
        var /= len(self.img)
        std = torch.sqrt(var)
        print(mean, std)

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        self.transform_image = transforms.ToTensor()

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        image_original = self.img[idx]
        label = self.labels[idx]
        places = self.places[idx]
        if self.split == 'test':
            return self.transform(image_original), self.transform_image(image_original), label, places
        return self.transform(image_original), label, places

In [ ]:
class BarlowTwins(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = torchvision.models.resnet34(zero_init_residual=True, weights=None)
        self.backbone.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.backbone.maxpool = nn.Identity()
        self.size_in = self.backbone.fc.in_features
        self.size_out = 256
        self.backbone.fc = nn.Identity()

        sizes = [self.size_in, self.size_out, self.size_out]
        layers = []
        for i in range(len(sizes) - 2):
            layers.append(nn.Linear(sizes[i], sizes[i + 1], bias=False))
            layers.append(nn.BatchNorm1d(sizes[i + 1]))
            layers.append(nn.ReLU(inplace=True))
        layers.append(nn.Linear(sizes[-2], sizes[-1], bias=False))
        self.projector = nn.Sequential(*layers)

    def forward(self, x, use_projector=True):
        if not use_projector:
            return self.backbone(x)
        z = self.projector(self.backbone(x))
        return z

In [ ]:
def generate_resnet_from_dict(path):
    model = BarlowTwins()
    model.load_state_dict(torch.load(path))
    return model.backbone, model.size_in

In [ ]:
from pytorch_grad_cam import HiResCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

image_map = {}
pt_list = range(40, 40+80*8+1, 120)
count_ocean = 10
count_forest = 10
for ratio in ratios:
    root_dir = f"/home/dataset/waterbird_single_background/{ratio}"
    metadata_path = f"/home/dataset/waterbird_single_background/{ratio}/metadata.csv"

    test_dataset = WaterbirdDataset(
        root_dir=root_dir,
        metadata_path=metadata_path,
        split='test',
        balance_classes=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=1,
        shuffle=False,
    )

    image_map[ratio] = []
    save_path = f"{directory_models}/{ratio}/linear"
    for i in range(len(pt_list)):
        print(f"Processing {pt_list[i]}")
        model, size_out = generate_resnet_from_dict(f"{directory_models}/{ratio}/{pt_list[i]}.pt")
        model = model.to(device)
        linear = nn.Linear(size_out, 2)
        linear.load_state_dict(torch.load(f"{directory_models}/{ratio}/linear/{pt_list[i]}.pt"))
        linear = linear.to(device)

        # target_layers = [model.layer1[-1], model.layer2[-1], model.layer3[-1], model.layer4[-1]]
        target_layers = [model.layer4[-1]]
        model_grad = nn.Sequential(model, linear).to(device)

        i_ocean = 0
        i_forest = 0
        image_org = []
        image_grad = []
        for index, (image_tensor, image, label, background) in enumerate(test_loader):
            if label == background:
                continue
            if i_ocean >= count_ocean and i_forest >= count_forest:
                break
            if label == 0 and i_ocean >= count_ocean:
                continue
            if label == 1 and i_forest >= count_forest:
                continue
            # targets = [ClassifierOutputTarget(1)]
            targets = None
            current_image = image[0, :, :, :]
            current_image = current_image.permute(1, 2, 0)
            current_image = current_image.cpu().numpy()
            with HiResCAM(model=model_grad, target_layers=target_layers) as cam:
                grayscale_cam = cam(input_tensor=image_tensor, targets=targets)
                grayscale_cam = grayscale_cam[0, :]
                visualization = show_cam_on_image(current_image, grayscale_cam, use_rgb=True)
                model_outputs = cam.outputs
            # plt.imshow(to_pil_image(current_image))
            # plt.show()
            # plt.imshow(to_pil_image(visualization))
            # plt.show()
            image_org.append(to_pil_image(current_image))
            image_grad.append(to_pil_image(visualization))
            # print(f"correct answer:{label.item()}, spurious answer: {background.item()}")
            # print(f"model: {model_outputs.detach().cpu()}")
            if label == 0:
                i_ocean += 1
            else:
                i_forest += 1

        if len(image_map[ratio]) == 0:
            image_map[ratio].append(image_org)
        image_map[ratio].append(image_grad)

In [ ]:
n_images = len(pt_list) - 1
print(len(ratios)*2, n_images)
scale = 0.75
fig, axes = plt.subplots(n_images, len(ratios)*2, figsize=(len(ratios)*2*scale, n_images*scale), sharey=True, sharex=True)
target_ocean = 7
target_forest = count_ocean + 0
for ratio_index in range(len(ratios)):
    current_images = image_map[ratios[ratio_index]]
    for j in range(n_images):
        axes[j, ratio_index].imshow(current_images[j][target_ocean])
        axes[j, ratio_index+len(ratios)].imshow(current_images[j][target_forest])
        axes[j, ratio_index+len(ratios)].axis('off')

        if ratio_index == 0:
            axes[j, 0].axes.get_xaxis().set_ticks([])
            axes[j, 0].axes.get_yaxis().set_ticks([])
            axes[j, 0].set_ylabel(f'Step {pt_list[j]}', rotation=0, labelpad=22, va='center', fontsize=8)
        else:
            axes[j, ratio_index].axis('off')

        if j == 0:
            axes[0, ratio_index].set_title(f'Ratio {ratios[ratio_index]:.3f}\nOcean', fontsize=8)
            axes[0, ratio_index+len(ratios)].set_title(f'Ratio {ratios[ratio_index]:.3f}\nForest', fontsize=8)

plt.subplots_adjust(wspace=0.02, hspace=0.02)
plt.tight_layout(pad=0.1)
plt.savefig("figure/gradcam.pdf")
plt.show()